In [ ]:
# Less Salt Less Sugar Monthly Report (launch from 31Aug2026. last for 1 year at least)
#     1. Daily Badge impression by device 
#     2. Daily Brand Page (LMS) Pageview/ Impressions     
#     3. Daily Theme Listing Impressions           --> search --> 少鹽少糖
 

In [57]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ2.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [58]:
current_date = datetime.date.today()
first_day_of_current_month = datetime.date(current_date.year, current_date.month, 1)
last_day_of_previous_month = first_day_of_current_month - datetime.timedelta(days=1)

month = last_day_of_previous_month.month
year = last_day_of_previous_month.year

str_month = str(month)
if len(str_month)==1:
    str_month = '0'+str_month
str_month

'08'

In [59]:
template_file = 'report_for_BD_template.xlsx'
excel_name = template_file.replace('template.xlsx', '%s.xlsx' % datetime.date.today())
copyfile(template_file, excel_name)

'report_for_BD_2026-09-23.xlsx'

In [ ]:
#少鹽少糖食店SR1 page view  (Not Need)

sql = f'''
with sr1 as
  (select '$$$少鹽少糖食店$$$' AS dummy, platform
  from `openrice-production.ORGA.PV_{year}{str_month}*`
  where -- eventcategory in ('Search Related', 'WebEvent')
   (lower((select item.value from unnest(eventlabel.list) where lower(item.param) = 'dedicatedpromotionid')) like '13' or 
       lower((select item.value from unnest(eventlabel.list) where lower(item.param) = 'amtid')) like '1093' or
       lower(eventdata) like '%hongkong%amenityid=1093%'))

select platform, count(1) from sr1
group by platform
    '''

df_big_query = client.query(sql).result().to_dataframe()

web = df_big_query.query("platform=='mobile' | platform=='desktop'  ").f0_.sum()
app = df_big_query.query("platform=='android' | platform=='ios' | platform=='hms' ").f0_.sum()
temp_df = pd.DataFrame({'date':[f'{year}-{str_month}'],'web':[web],'app':[app]})

with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    temp_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=2, header=None, index=False)

In [60]:
# 1. Daily Badge impression by device     DONE
#  --> monthly_icon_impression_report_result.ipynb cell 8 少鹽少糖食店 Icon Impression v2
# 
# #少鹽少糖食店 Icon Impression v2

sql = f'''

SELECT date(time) as querydate,platform,count(1) as count_ FROM `openrice-production.ORGA.PV_{year}{str_month}*` 
WHERE EventAction = 'impression.poi'
and cast(REGEXP_EXTRACT(lower(EventLabelRaw), r'poiid:(\d+)') as INT64) in  (Select poiid FROM `openrice-production.openrice3.promotionpoi` WHERE PromotionId =11)
group by platform,querydate
    '''

df_big_query = client.query(sql).result().to_dataframe()

pivoted_df = df_big_query.pivot(index='querydate', columns='platform', values='count_')
pivoted_df = pivoted_df.fillna(0)
pivoted_df["Web"]=pivoted_df["desktop"]
pivoted_df["Mobile Web"]=pivoted_df["mobile"]
try:
    pivoted_df["Android"]=pivoted_df["android"]+pivoted_df["hms"]
except:
    pivoted_df["Android"]=pivoted_df["android"]
    
pivoted_df["IOS"]=pivoted_df["ios"]
pivoted_df =pivoted_df[["Web","Mobile Web","Android","IOS"]]

pivoted_df.index.name = None
pivoted_df.reset_index(inplace=True)

with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    pivoted_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=7, header=None, index=False)

<>:12: SyntaxWarning: invalid escape sequence '\d'
<>:12: SyntaxWarning: invalid escape sequence '\d'
C:\Users\lenalee\AppData\Local\Temp\ipykernel_8656\2701819859.py:12: SyntaxWarning: invalid escape sequence '\d'
  '''
C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
# 2. Daily Brand Page (LMS) Pageview        
# 進入search頁面後，點擊少鹽少糖食店按鈕
# 17:35:05|| or.search.layer.search| CityID:0;geo:22.2915336%2C114.2081752;LndID:35336;Page:1;sr:lmsSr1;Lang:zh_TW;Ver:7.20.4; sn:hk.Search.layer
# unique users; pageview: total click count
# by Web / Mobile Web / Android / iOS  --> LSLS_badge.png (monthly_icon_impression_report_result.ipynb cell 8) for definition

sql = """
SELECT
    date(time) as querydate,
    platform,
    count(1) as count
FROM `openrice-production.ORGA.PV_{year}{str_month}*`
WHERE LOWER(EventAction) LIKE '%or.search.layer.search%'
    AND LOWER(EventLabelRaw) LIKE '%sr:lmssr1%'
GROUP BY 1, 2
"""

In [40]:
# ADV Search
# 3. Daily Theme Listing Impressions --> adv search
# 進入少鹽少糖食店頁面後，點擊篩選搜尋按鈕
# 18:08:25|| or.search.adv| CityID:0;geo:22.2915758%2C114.2081138;LndID:35336;Page:1;sr:lmsSr1;Lang:zh_TW;Ver:7.20.4; sn:hk.AdvSearch
# by Web / Mobile Web / Android / iOS  --> LSLS_badge.png (monthly_icon_impression_report_result.ipynb cell 8) for definition

sql = f"""
with adv_search as (
    select '$$$少鹽少糖食店$$$' AS dummy, platform
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (LOWER(EventAction) = 'or.search.adv'
    AND LOWER(EventLabelRaw) LIKE '%lms%')
)

select platform, count(1) as count
from adv_search
group by platform
"""

df_big_query_5 = client.query(sql).result().to_dataframe()

df_big_query_5


# web = df_big_query_5.query("platform=='mobile' | platform=='desktop'  ")["count"].sum()
# app = df_big_query_5.query("platform=='android' | platform=='ios' | platform=='hms' ")["count"].sum()
# temp_df = pd.DataFrame({'date':[f'{year}-{str_month}'],'web':[web],'app':[app]})

# with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
# #     book = load_workbook(excel_name)
# #     writer.book = book
# #     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
#     temp_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=42, header=None, index=False)


C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,platform,count
0,ios,447
1,hms,952
2,android,29538


In [62]:
check_sql = f"""
SELECT *
FROM `openrice-production.ORGA.PV_{year}{str_month}*`
WHERE platform IN ('desktop', 'mobile')
AND LOWER(EventAction) LIKE '%or.search.layer.search%'
ORDER BY time DESC
LIMIT 100
"""

web_lms_records = client.query(check_sql).result().to_dataframe()
web_lms_records



C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,SessionId,DeviceId,Time,UserId,IP,EventAction,EventCategory,EventData,EventEntity,EventLabel,EventLabelRaw,EventSource,UserAgent,Product,MachineName,CollectTime,Platform
0,1455529,ce9dbf3a-c079-4f09-ac3a-4df9d6f43cce,2026-08-31 23:59:59.300000+00:00,,14.0.228.23,or.search.layer.search,WebEvent,https://www.openrice.com/zh/hongkong/restauran...,,"{'list': [{'item': {'List': 0, 'Param': 'whatw...",whatwhere:生蠔任食;Sn:https://www.openrice.com/zh/...,,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 23:39:01+00:00,mobile
1,1455529,ce9dbf3a-c079-4f09-ac3a-4df9d6f43cce,2026-08-31 23:59:55.800000+00:00,,14.0.228.23,or.search.layer.search,WebEvent,https://www.openrice.com/zh/hongkong/restauran...,,"{'list': [{'item': {'List': 0, 'Param': 'whatw...",whatwhere:生蠔任食;Sn:https://www.openrice.com/zh/...,,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 23:39:01+00:00,mobile
2,1426952,ca9a4da3-a897-4ab9-aaf5-123d201b1901,2026-08-31 23:56:10.200000+00:00,,112.119.66.126,or.search.layer.search,WebEvent,https://www.openrice.com/zh/hongkong/restauran...,,"{'list': [{'item': {'List': 0, 'Param': 'Sn', ...",Sn:https://www.openrice.com/zh/hongkong/restau...,,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 23:35:02+00:00,mobile
3,1426952,ca9a4da3-a897-4ab9-aaf5-123d201b1901,2026-08-31 23:56:09.100000+00:00,,112.119.66.126,or.search.layer.search,WebEvent,https://www.openrice.com/zh/hongkong/restauran...,,"{'list': [{'item': {'List': 0, 'Param': 'Sn', ...",Sn:https://www.openrice.com/zh/hongkong/restau...,,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 23:35:02+00:00,mobile
4,1320293,bb71e878-47a2-4bba-a43b-0a2ada4ab945,2026-08-31 23:53:19.300000+00:00,,180.188.175.238,or.search.layer.search,WebEvent,https://www.openrice.com/zh/hongkong/explore/r...,,"{'list': [{'item': {'List': 0, 'Param': 'whatw...",whatwhere:灣仔;Sn:https://www.openrice.com/zh/ho...,,Mozilla/5.0 (iPhone; CPU iPhone OS 17_0_3 like...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 23:33:02+00:00,mobile
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,110507,0f8f3489-45d2-4079-a360-700306524ef2,2026-08-31 22:15:14.100000+00:00,,58.82.250.85,or.search.layer.search,WebEvent,https://www.openrice.com/zh/hongkong/restauran...,,"{'list': [{'item': {'List': 0, 'Param': 'whatw...",whatwhere:觀塘 台灣菜;Sn:https://www.openrice.com/z...,,Mozilla/5.0 (iPhone; CPU iPhone OS 18_3_2 like...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 21:54:02+00:00,mobile
96,766787,6ce124d3-a5db-4921-95fc-0817b52aca91,2026-08-31 22:15:10.600000+00:00,,182.239.87.215,or.search.layer.search,WebEvent,https://www.openrice.com/zh/hongkong/restauran...,,"{'list': [{'item': {'List': 0, 'Param': 'whatw...",whatwhere:AI 劇;Sn:https://www.openrice.com/zh/...,,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 21:54:02+00:00,mobile
97,1581753,e08c2b98-19c2-4194-b83a-a3f555004622,2026-08-31 22:11:46.200000+00:00,,203.218.36.211,or.search.layer.search,WebEvent,https://www.openrice.com/zh/hongkong/restaurants,,"{'list': [{'item': {'List': 0, 'Param': 'whatw...",whatwhere:Airside;Sn:https://www.openrice.com/...,,Mozilla/5.0 (iPhone; CPU iPhone OS 18_7 like M...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-08-31 21:51:02+00:00,mobile
98,1521578,d8051c05-6036-44fa-88cb-f78f4d741666,2026-08-31 22:11:10.300000+00:00,,182.239.87.4,or.search.layer.search,WebEvent,https://www.openrice.com/zh/hongkong/restauran...,,"{'list': [{'item': {'List': 0, 'Param': 'distr...",districtId:2010;dishId:1076;Sn:https://www.ope...,,Mozilla/5.0 (Linux; Android 10; K) AppleWebKit...,OpenRice,production-orga-openrice-netcore

In [63]:
web_lms_records.to_csv("web_records.csv", index=False)

In [53]:
# 3. Daily Theme Listing Impressions           no date, group by web & app
# 進入少鹽少糖食店頁面後，點擊篩選搜尋按鈕
# 17:37:14|| or.search.layer.search| CityID:0;geo:22.2915161%2C114.2081815;LndID:35336;Page:1;sr:lmsSr1;Lang:zh_TW;Ver:7.20.4; sn:hk.Search.layer
# 17:37:14|| view.SR1.Promotion| CityID:0;PromotionID:13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C12%2C13%2C13%2C13%2C13%2C13%2C13;;Lang:zh_TW;Ver:7.20.4; sn:hk.LMS2.35336.tab.-990.1
# by Web / Mobile Web / Android / iOS  --> LSLS_badge.png (monthly_icon_impression_report_result.ipynb cell 8) for definition

sql = f"""
with theme_list as (
    select platform
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (LOWER(EventAction) = 'view.sr1.promotion'
    AND ((LOWER(EventLabelRaw) LIKE '%lms2%') or (LOWER(EventLabelRaw) LIKE '%promotionid:13%')))
)

select platform, count(1) as count
from theme_list
group by platform
"""

df_big_query_3 = client.query(sql).result().to_dataframe()

#df_big_query_3


web = df_big_query_3.query("platform=='mobile' | platform=='desktop'  ")["count"].sum()
app = df_big_query_3.query("platform=='android' | platform=='ios' | platform=='hms' ")["count"].sum()
temp_df = pd.DataFrame({'date':[f'{year}-{str_month}'],'web':[web],'app':[app]})

temp_df

# with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
# #     book = load_workbook(excel_name)
# #     writer.book = book
# #     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
#     temp_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=42, header=None, index=False)


C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,date,web,app
0,2026-08,444516,3494588


In [56]:
df_big_query_3

,platform,count
0,bot,12047
1,tablet,1483
2,hms,9306
3,android,695662
4,smart tv,11
5,mobile,269304
6,unknown,73
7,desktop,175212
8,ios,2789620


In [51]:
# 3. Daily Theme Listing Impressions   hv date, group by devices

sql = f"""
    select date(time) as querydate, platform, count(1) as count
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (
        LOWER(EventAction) = 'view.sr1.promotion'
        AND (
            (LOWER(EventLabelRaw) LIKE '%lms2%') or (LOWER(EventLabelRaw) LIKE '%promotionid:13%')
            )
        )
    group by platform, querydate
    """


df_big_query_6 = client.query(sql).result().to_dataframe()

pivoted_df = df_big_query_6.pivot(index='querydate', columns='platform', values='count')
pivoted_df = pivoted_df.fillna(0)
pivoted_df["Web"]=pivoted_df["desktop"]
pivoted_df["Mobile Web"]=pivoted_df["mobile"]
try:
    pivoted_df["Android"]=pivoted_df["android"]+pivoted_df["hms"]
except:
    pivoted_df["Android"]=pivoted_df["android"]
    
pivoted_df["IOS"]=pivoted_df["ios"]
pivoted_df =pivoted_df[["Web","Mobile Web","Android","IOS"]]

pivoted_df.index.name = None
pivoted_df.reset_index(inplace=True)

pivoted_df
# with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
# #     book = load_workbook(excel_name)
# #     writer.book = book
# #     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
#     pivoted_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=7, header=None, index=False)

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


platform,index,Web,Mobile Web,Android,IOS
0,2026-08-01,6414,15848,47294,183385
1,2026-08-02,5632,17436,44964,176448
2,2026-08-03,13249,15358,34594,132276
3,2026-08-04,15280,24456,37118,145272
4,2026-08-05,15018,28123,37799,147249
5,2026-08-06,15382,28674,38881,152241
6,2026-08-07,16652,22031,42297,167167
7,2026-08-08,6,425,24150,82287
8,2026-08-09,0,272,18860,68328
9,2026-08-10,13,146,7563,27749
